# Optimized Whisper on AWS Neuron (Trainium2 / Inferentia2)

This notebook runs OpenAI's [Whisper large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) on AWS Neuron instances using [NxD Inference](https://github.com/aws-neuron/neuronx-distributed-inference) with four optimizations:

1. **Cross-attention K/V cache** — eliminates redundant K/V projections (19.7B FLOPs/token) and audio transfer (3.84MB/token) during decode
2. **Fused QKV projections** — replaces 3 separate Q/K/V linear layers with a single fused matmul in self-attention
3. **NKI flash attention** — replaces matmul-based attention with hardware-accelerated flash attention in all 32 encoder layers
4. **NKI fused Conv1D+GELU** — replaces separate Conv1D and GELU ops with a single fused NKI kernel in the encoder frontend

**Supported instances:** trn2.3xlarge, inf2.xlarge (or any Neuron instance with ≥1 NeuronCore)

**Model:** whisper-large-v3-turbo (809M params, 32 encoder layers, 4 decoder layers) at TP=1, FP16

## 1. Setup

Activate the pre-installed PyTorch Neuron environment and install dependencies.

```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
```

In [ ]:
!pip install -q openai-whisper soundfile librosa gtts jiwer 2>/dev/null
!sudo add-apt-repository -y universe > /dev/null 2>&1 && sudo apt-get update -qq > /dev/null 2>&1 && sudo apt-get install -y -qq ffmpeg > /dev/null 2>&1
!which ffmpeg

## 2. Install Optimized NxDI

Clone our optimized fork of neuronx-distributed-inference and install it over the system package.
This replaces only the Whisper model files — all other NxDI models are unaffected.

**Note:** If running via `run_notebook.sh`, these packages are pre-installed. The cells below
detect this and skip the install if already present.

In [ ]:
import os
import subprocess
import sys

NXDI_BRANCH = "fix/whisper-all-optimizations"
NXDI_REPO = "https://github.com/jimburtoft/neuronx-distributed-inference.git"
NXDI_CLONE_DIR = "/tmp/nxdi-optimized"

# Clone the optimized branch
if not os.path.exists(NXDI_CLONE_DIR):
    subprocess.run(
        ["git", "clone", "--branch", NXDI_BRANCH, "--single-branch", "--depth", "1", NXDI_REPO, NXDI_CLONE_DIR],
        check=True,
    )
    print(f"Cloned {NXDI_BRANCH} to {NXDI_CLONE_DIR}")
else:
    print(f"Already cloned at {NXDI_CLONE_DIR}")

# Install over the system NxDI package (if not already installed from run_notebook.sh)
try:
    import neuronx_distributed_inference
    print(f"NxDI already available: {neuronx_distributed_inference.__file__}")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", NXDI_CLONE_DIR], check=True)
    print("Optimized NxDI installed — please restart kernel if running interactively")

## 3. Install NKI Conv1D kernel (optional)

The DLAMI includes `nkilib` but may lack the newer `conv1d` kernel. We copy it
from the nki-library repo. If unavailable, the notebook falls back to PyTorch Conv1D.

In [ ]:
import os, subprocess, sys

NKI_LIB_DIR = "/tmp/nki-library"

try:
    from nkilib.experimental.conv.conv1d import conv1d as nki_conv1d
    print("NKI Conv1D kernel already available")
except ImportError:
    # Copy conv1d.py from nki-library repo into the installed nkilib package
    try:
        import nkilib.experimental.conv
        conv_dir = os.path.dirname(nkilib.experimental.conv.__file__)
        if not os.path.exists(NKI_LIB_DIR):
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/aws-neuron/nki-library.git", NKI_LIB_DIR],
                check=True, capture_output=True)
        src = os.path.join(NKI_LIB_DIR, "src", "nkilib_src", "nkilib", "experimental", "conv", "conv1d.py")
        dst = os.path.join(conv_dir, "conv1d.py")
        import shutil
        shutil.copy2(src, dst)
        from nkilib.experimental.conv.conv1d import conv1d as nki_conv1d
        print(f"NKI Conv1D kernel installed to {dst}")
    except Exception as e:
        print(f"NKI Conv1D not available ({e}) — will use PyTorch fallback")

## 4. Detect Hardware

Auto-detect whether we're on Trainium or Inferentia and configure accordingly.

In [ ]:
import time
import json
import torch

os.environ["NEURON_RT_NUM_CORES"] = "1"

# NKI conv1d kernel needs platform hint during compilation
result = subprocess.run(["neuron-ls"], capture_output=True, text=True)
neuron_ls_output = result.stdout
print(neuron_ls_output)

if "trn2" in neuron_ls_output.lower():
    PLATFORM = "trn2"
    os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"
elif "trn1" in neuron_ls_output.lower():
    PLATFORM = "trn1"
elif "inf2" in neuron_ls_output.lower() or "inf1" in neuron_ls_output.lower():
    PLATFORM = "inf2"
else:
    # Fallback: check instance type from metadata
    try:
        token_result = subprocess.run(
            ["curl", "-s", "-X", "PUT", "http://169.254.169.254/latest/api/token",
             "-H", "X-aws-ec2-metadata-token-ttl-seconds: 60"],
            capture_output=True, text=True, timeout=2
        )
        token = token_result.stdout.strip()
        itype_result = subprocess.run(
            ["curl", "-s", "-H", f"X-aws-ec2-metadata-token: {token}",
             "http://169.254.169.254/latest/meta-data/instance-type"],
            capture_output=True, text=True, timeout=2
        )
        instance_type = itype_result.stdout.strip()
        if "trn2" in instance_type:
            PLATFORM = "trn2"
            os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"
        elif "trn1" in instance_type:
            PLATFORM = "trn1"
        elif "inf" in instance_type:
            PLATFORM = "inf2"
        else:
            PLATFORM = "unknown"
    except Exception:
        PLATFORM = "unknown"

print(f"\nDetected platform: {PLATFORM}")

In [ ]:
from neuronx_distributed_inference.models.config import NeuronConfig
from neuronx_distributed_inference.models.whisper.modeling_whisper import (
    WhisperInferenceConfig,
    NeuronApplicationWhisper,
)
from neuronx_distributed_inference.utils.hf_adapter import load_pretrained_config

## 5. Download Model

In [ ]:
MODEL_PATH = "/tmp/whisper-large-v3-turbo/"
COMPILED_MODEL_PATH = f"/tmp/whisper-large-v3-turbo-neuron-{PLATFORM}/"

if not os.path.exists(os.path.join(MODEL_PATH, "config.json")):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="openai/whisper-large-v3-turbo",
        local_dir=MODEL_PATH,
        ignore_patterns=["*.onnx", "*.tflite", "flax_model*", "tf_model*"],
    )
    print(f"Downloaded to {MODEL_PATH}")
else:
    print(f"Model already exists at {MODEL_PATH}")

## 6. Compile for Neuron

NxDI compiles three separate graphs:
- **Encoder** — processes the 30-second mel spectrogram window
- **Decoder Prefill** — processes all prompt tokens at once, caches encoder K/V
- **Decoder Decode** — generates one token at a time using KV cache

Compilation is a one-time cost. The compiled model is platform-specific (trn2 vs inf2).

In [ ]:
neuron_config = NeuronConfig(
    batch_size=1,
    torch_dtype=torch.float16,
    tp_degree=1,
)
inference_config = WhisperInferenceConfig(
    neuron_config,
    load_config=load_pretrained_config(MODEL_PATH),
)

if not os.path.exists(COMPILED_MODEL_PATH):
    print(f"Compiling for {PLATFORM} (one-time cost, ~60-120s)...")
    t0 = time.time()
    model = NeuronApplicationWhisper(MODEL_PATH, config=inference_config)
    model.compile(COMPILED_MODEL_PATH)
    compile_time = time.time() - t0
    print(f"Compilation complete in {compile_time:.1f}s")
else:
    compile_time = None
    print(f"Using cached compiled model at {COMPILED_MODEL_PATH}")

## 7. Load Compiled Model

In [ ]:
t0 = time.time()
model = NeuronApplicationWhisper(COMPILED_MODEL_PATH, config=inference_config)
model.load(COMPILED_MODEL_PATH)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")

## 8. Generate Test Audio

Create three speech samples of different lengths using Google Text-to-Speech.

In [ ]:
import shutil
import soundfile as sf
import numpy as np
from gtts import gTTS

FFMPEG = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
AUDIO_DIR = "/tmp/whisper_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

# Three test samples of increasing length
samples = {
    "short": (
        "The quick brown fox jumps over the lazy dog. "
        "This is a test of automatic speech recognition."
    ),
    "medium": (
        "Artificial intelligence has transformed the way we interact with technology. "
        "Machine learning models can now understand and generate human language with remarkable accuracy. "
        "Speech recognition systems like Whisper have made it possible to transcribe audio in real time, "
        "supporting dozens of languages and handling noisy environments with ease. "
        "The combination of hardware acceleration and optimized software makes these capabilities "
        "accessible at scale, enabling applications from live captioning to voice assistants."
    ),
    "long": (
        "The history of artificial intelligence is a fascinating journey through decades of research and discovery. "
        "In the nineteen fifties, pioneers like Alan Turing asked whether machines could think, "
        "laying the philosophical groundwork for an entirely new field of computer science. "
        "Early systems used symbolic reasoning and hand-crafted rules to solve narrowly defined problems. "
        "The nineteen eighties saw the rise of expert systems, which encoded human knowledge into decision trees. "
        "But it was the advent of deep learning in the twenty tens that truly revolutionized the field. "
        "Neural networks with millions of parameters could learn directly from data, "
        "outperforming traditional approaches on tasks from image classification to natural language processing. "
        "Today, large language models and multimodal systems represent the cutting edge, "
        "capable of understanding text, images, and speech simultaneously. "
        "Custom silicon like AWS Trainium and Inferentia accelerates these workloads, "
        "making advanced AI accessible and cost-effective for organizations of every size."
    ),
}

audio_files = {}
for name, text in samples.items():
    wav_path = os.path.join(AUDIO_DIR, f"{name}.wav")
    if not os.path.exists(wav_path):
        mp3_path = os.path.join(AUDIO_DIR, f"{name}.mp3")
        tts = gTTS(text, lang="en")
        tts.save(mp3_path)
        subprocess.run(
            [FFMPEG, "-y", "-i", mp3_path, "-ar", "16000", "-ac", "1", wav_path],
            capture_output=True, check=True,
        )
    data, sr = sf.read(wav_path)
    duration = len(data) / sr
    audio_files[name] = {"path": wav_path, "duration": duration, "reference": text}
    print(f"{name:7s}: {duration:.1f}s — {wav_path}")

## 9. Transcribe

Run transcription on all test samples. The NxDI model exposes the same `transcribe()` API as OpenAI Whisper.

In [ ]:
# Warmup run
_ = model.transcribe(audio_files["short"]["path"], language="en", verbose=False)

for name, info in audio_files.items():
    result = model.transcribe(info["path"], language="en", verbose=False)
    info["neuron_text"] = result["text"].strip()
    print(f"\n--- {name} ({info['duration']:.1f}s) ---")
    print(f"Neuron: {info['neuron_text']}")

## 10. Benchmark

Measure latency across 10 runs per audio sample.

In [ ]:
NUM_RUNS = 10

for name, info in audio_files.items():
    latencies = []
    for _ in range(NUM_RUNS):
        t0 = time.time()
        _ = model.transcribe(info["path"], language="en", verbose=False)
        latencies.append(time.time() - t0)
    info["neuron_avg_latency"] = np.mean(latencies)
    info["neuron_min_latency"] = min(latencies)
    info["neuron_rtf"] = info["duration"] / info["neuron_avg_latency"]

print(f"{'Audio':<8} {'Duration':>8} {'Avg Latency':>12} {'Min Latency':>12} {'RTF':>6}")
print("-" * 50)
for name, info in audio_files.items():
    print(f"{name:<8} {info['duration']:>7.1f}s {info['neuron_avg_latency']:>11.3f}s {info['neuron_min_latency']:>11.3f}s {info['neuron_rtf']:>5.1f}x")

## 11. CPU Comparison & Accuracy

Run the same audio through OpenAI Whisper on CPU. Compare transcription accuracy using Word Error Rate (WER).

In [ ]:
import whisper
from jiwer import wer

cpu_model = whisper.load_model("large-v3-turbo", device="cpu")

for name, info in audio_files.items():
    t0 = time.time()
    cpu_result = cpu_model.transcribe(info["path"], language="en", verbose=False)
    info["cpu_latency"] = time.time() - t0
    info["cpu_text"] = cpu_result["text"].strip()
    info["speedup"] = info["cpu_latency"] / info["neuron_avg_latency"]
    info["wer"] = wer(info["cpu_text"], info["neuron_text"])

print(f"{'Audio':<8} {'Neuron':>10} {'CPU':>10} {'Speedup':>8} {'WER':>6}")
print("-" * 46)
for name, info in audio_files.items():
    print(f"{name:<8} {info['neuron_avg_latency']:>9.3f}s {info['cpu_latency']:>9.3f}s {info['speedup']:>7.1f}x {info['wer']:>5.1%}")

print("\n--- Transcription Comparison ---")
for name, info in audio_files.items():
    match = "MATCH" if info["neuron_text"] == info["cpu_text"] else "DIFF"
    print(f"\n[{name}] ({match})")
    print(f"  Neuron: {info['neuron_text']}")
    print(f"  CPU:    {info['cpu_text']}")

## 12. Results Summary

In [ ]:
# Collect all results into a summary dict
results_summary = {
    "platform": PLATFORM,
    "model": "openai/whisper-large-v3-turbo",
    "precision": "FP16",
    "tp_degree": 1,
    "compile_time_s": compile_time,
    "load_time_s": round(load_time, 1),
    "benchmarks": {},
}

for name, info in audio_files.items():
    results_summary["benchmarks"][name] = {
        "audio_duration_s": round(info["duration"], 1),
        "neuron_avg_latency_s": round(info["neuron_avg_latency"], 3),
        "neuron_min_latency_s": round(info["neuron_min_latency"], 3),
        "rtf": round(info["neuron_rtf"], 1),
        "cpu_latency_s": round(info["cpu_latency"], 3),
        "speedup_vs_cpu": round(info["speedup"], 1),
        "wer_vs_cpu": round(info["wer"], 4),
        "neuron_text": info["neuron_text"],
        "cpu_text": info["cpu_text"],
    }

# Save to JSON
results_file = f"/tmp/whisper_results_{PLATFORM}.json"
with open(results_file, "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"Results saved to {results_file}")
print(json.dumps(results_summary, indent=2))

## 13. Transcribe Your Own Audio

Replace the path below with any audio file (WAV, MP3, FLAC, etc.).

In [ ]:
# my_audio = "/path/to/your/audio.wav"
# result = model.transcribe(my_audio, language="en", verbose=True)
# print(result["text"])

## 14. LNC=1 Benchmark (trn2 only)

On trn2.3xlarge, the **Logical NeuronCore (LNC)** configuration determines how physical
cores are split:

| LNC | Logical Cores | HBM per Core | Default |
|-----|--------------|-------------|----------|
| LNC=2 | 4 cores | 24 GB each | Yes |
| LNC=1 | 8 cores | 12 GB each | No |

For small models like Whisper (809M params, TP=1), LNC=2 gives each core more HBM
bandwidth. But LNC=1 gives more cores for data parallelism. The question is whether
LNC=1 single-stream latency is competitive with LNC=2.

**This section requires a reboot** to change LNC mode, so it cannot run inline in the
notebook. Run the cells below to set up, then follow the manual steps.

**Skip this section** if you are on inf2 or trn1 (LNC only applies to trn2).

In [ ]:
# Check current LNC mode
if PLATFORM == 'trn2':
    import subprocess
    result = subprocess.run(['neuron-ls'], capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    # Count NeuronCore entries
    nc_count = sum(1 for line in lines if 'NeuronCore' in line or line.strip().startswith(('0', '1', '2', '3', '4', '5', '6', '7')))
    print(result.stdout)
    print(f'\nDetected ~{nc_count} logical cores')
    if nc_count <= 4:
        print('Currently in LNC=2 mode (4 cores, 24GB HBM each)')
        print('\nTo switch to LNC=1 for benchmarking:')
        print('  1. Save this notebook')
        print('  2. In a terminal, run:')
        print('     echo NEURON_LOGICAL_NC_CONFIG=1 | sudo tee /etc/environment')
        print('     sudo reboot')
        print('  3. After reboot, re-run this notebook from the top')
        print('     (compilation will be re-done for the new LNC mode)')
    else:
        print('Currently in LNC=1 mode (8 cores, 12GB HBM each)')
        print('The benchmark above already ran in LNC=1 mode.')
else:
    print(f'Platform is {PLATFORM}, not trn2 — LNC does not apply. Skip this section.')

### Automated LNC=1 Benchmark Script

Alternatively, run this self-contained script from a terminal. It sets LNC=1 for the
current process, recompiles, and benchmarks without needing a reboot:

```bash
# From the Neuron venv:
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
NEURON_LOGICAL_NC_CONFIG=1 python /tmp/lnc1_benchmark.py
```

The cell below writes the benchmark script.

In [ ]:
lnc1_script = '''#!/usr/bin/env python3
"""LNC=1 Whisper benchmark for trn2.3xlarge.

Run with: NEURON_LOGICAL_NC_CONFIG=1 python lnc1_benchmark.py

This recompiles the model for the LNC=1 core layout and runs the same
benchmark as the notebook. Results are saved to /tmp/whisper_results_trn2_lnc1.json.
"""
import os, sys, time, json, subprocess
import numpy as np

# Verify we are in LNC=1 mode
lnc = os.environ.get("NEURON_LOGICAL_NC_CONFIG", "2")
if lnc != "1":
    print("ERROR: NEURON_LOGICAL_NC_CONFIG is not set to 1.")
    print("Run with: NEURON_LOGICAL_NC_CONFIG=1 python lnc1_benchmark.py")
    sys.exit(1)

result = subprocess.run(["neuron-ls"], capture_output=True, text=True)
print(result.stdout)

os.environ["NEURON_RT_NUM_CORES"] = "1"
os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"

import torch
from neuronx_distributed_inference.models.config import NeuronConfig
from neuronx_distributed_inference.models.whisper.modeling_whisper import (
    WhisperInferenceConfig, NeuronApplicationWhisper,
)
from neuronx_distributed_inference.utils.hf_adapter import load_pretrained_config

MODEL_PATH = "/tmp/whisper-large-v3-turbo/"
COMPILED_PATH = "/tmp/whisper-large-v3-turbo-neuron-trn2-lnc1/"

neuron_config = NeuronConfig(batch_size=1, torch_dtype=torch.float16, tp_degree=1)
inference_config = WhisperInferenceConfig(neuron_config, load_config=load_pretrained_config(MODEL_PATH))

# Compile
if not os.path.exists(COMPILED_PATH):
    print("Compiling for LNC=1...")
    t0 = time.time()
    model = NeuronApplicationWhisper(MODEL_PATH, config=inference_config)
    model.compile(COMPILED_PATH)
    compile_time = time.time() - t0
    print(f"Compilation: {compile_time:.1f}s")
else:
    compile_time = None
    print(f"Using cached: {COMPILED_PATH}")

# Load
t0 = time.time()
model = NeuronApplicationWhisper(COMPILED_PATH, config=inference_config)
model.load(COMPILED_PATH)
load_time = time.time() - t0
print(f"Load: {load_time:.1f}s")

# Benchmark
import soundfile as sf
AUDIO_DIR = "/tmp/whisper_audio"
NUM_RUNS = 10

audio_files = {}
for name in ["short", "medium", "long"]:
    wav_path = os.path.join(AUDIO_DIR, f"{name}.wav")
    data, sr = sf.read(wav_path)
    audio_files[name] = {"path": wav_path, "duration": len(data) / sr}

# Warmup
_ = model.transcribe(audio_files["short"]["path"], language="en", verbose=False)

results = {"lnc": 1, "compile_time_s": compile_time, "load_time_s": round(load_time, 1), "benchmarks": {}}

for name, info in audio_files.items():
    latencies = []
    for _ in range(NUM_RUNS):
        t0 = time.time()
        r = model.transcribe(info["path"], language="en", verbose=False)
        latencies.append(time.time() - t0)
    avg = np.mean(latencies)
    results["benchmarks"][name] = {
        "audio_duration_s": round(info["duration"], 1),
        "neuron_avg_latency_s": round(avg, 3),
        "neuron_min_latency_s": round(min(latencies), 3),
        "rtf": round(info["duration"] / avg, 1),
        "text": r["text"].strip(),
    }
    print(f"{name:8s} {info[\"duration\"]:6.1f}s  avg={avg:.3f}s  min={min(latencies):.3f}s  RTF={info[\"duration\"] / avg:.1f}x")

out = "/tmp/whisper_results_trn2_lnc1.json"
with open(out, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {out}")
'''

if PLATFORM == 'trn2':
    with open('/tmp/lnc1_benchmark.py', 'w') as f:
        f.write(lnc1_script)
    print('Wrote /tmp/lnc1_benchmark.py')
    print('Run with: NEURON_LOGICAL_NC_CONFIG=1 python /tmp/lnc1_benchmark.py')
else:
    print(f'Skipping — not on trn2 (platform={PLATFORM})')

## Performance Results

### trn2.3xlarge (LNC=2, SDK 2.28)

| Audio | Duration | Neuron Latency | RTF | CPU Latency | Speedup | WER |
|-------|----------|---------------|-----|-------------|---------|-----|
| short | 7.3s | 0.176s | 41.6x | 5.654s | 32.1x | 0.0% |
| medium | 38.9s | 0.459s | 84.7x | 5.948s | 12.9x | 0.0% |
| long | 83.0s | 4.272s | 19.4x | 35.538s | 8.3x | 32.7%* |

- Compilation: 27.9s
- Model load: 7.7s

### inf2.xlarge (SDK 2.28)

| Audio | Duration | Neuron Latency | RTF | CPU Latency | Speedup | WER |
|-------|----------|---------------|-----|-------------|---------|-----|
| short | 6.9s | 0.281s | 24.6x | 16.366s | 58.2x | 0.0% |
| medium | 36.5s | 2.314s | 15.8x | 27.984s | 12.1x | 2.9%* |
| long | 76.7s | 6.428s | 11.9x | 155.22s | 24.1x | 25.8%* |

- Compilation: 38.1s
- Model load: 11.4s

*\*WER note: High WER on long audio is caused by Whisper hallucinating gibberish after speech ends — a known model behavior, not a Neuron issue. Both Neuron and CPU produce hallucinations but with different text, inflating WER. The actual speech content is transcribed correctly on both.*

### trn2.3xlarge LNC=1 vs LNC=2

| Audio | LNC=2 Latency | LNC=1 Latency | Ratio | Verdict |
|-------|-------------|-------------|-------|----------|
| short | 0.176s | 0.231s | 1.31x | LNC=1 competitive |
| medium | 0.459s | 0.616s | 1.34x | LNC=1 competitive |
| long | 4.272s | 4.511s | 1.06x | LNC=1 competitive |

**Conclusion:** LNC=1 single-stream latency is only 1.06-1.34x slower than LNC=2, well below the 2x threshold. With LNC=1 providing 8 cores vs 4, a DP=8 deployment would achieve **~50% higher aggregate throughput** than DP=4 at LNC=2 for this model.

**Note:** Compiling for LNC=1 requires adding `--lnc=1` to the Neuron compiler flags. The current NxDI Whisper model does not pass this flag automatically — a patch to `get_compiler_args()` in `modeling_whisper.py` is needed.

### Optimizations Applied

| Optimization | Description | Impact |
|---|---|---|
| Cross-attn K/V cache | Skip redundant K/V projections during decode | ~2.5x decode latency reduction |
| Fused QKV | Single matmul for Q/K/V in self-attention | Reduced kernel launch overhead |
| NKI flash attention | Hardware flash attention in 32 encoder layers | 45% faster compilation |
| NKI fused Conv1D+GELU | Fused encoder frontend convolutions | Marginal (encoder frontend not bottleneck) |

All NKI kernels are from existing libraries (NxDI SDK and nki-library) — no custom kernels.